# 5 · Split Dataset into Train / Eval Sets
Splits by unique experiment (`FileName`) so no experiment leaks across splits.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
ANN_PARQUET = "./metadata/parquet_files/df_ann.parquet"
TRAIN_OUT   = "./metadata/parquet_files/train_ue.parquet"
EVAL_OUT    = "./metadata/parquet_files/eval_ue.parquet"
EVAL_FRAC   = 0.15   # fraction of experiments held out for evaluation
SEED        = 37

In [2]:
import numpy as np
import pandas as pd

## 1 · Load annotations

In [3]:
df_ann = pd.read_parquet(ANN_PARQUET)
print(f"Total particles: {len(df_ann):,}")

Total particles: 37,623,123


## 2 · Compute train / eval experiment indices

In [4]:
np.random.seed(SEED)
unique_exp = np.unique(df_ann['FileName'].values)
n          = len(unique_exp)
n_eval     = int(n * EVAL_FRAC)
unique_exp_idx = np.arange(n)

np.random.shuffle(unique_exp_idx)
eval_exp_idxs   = np.sort(unique_exp_idx[:n_eval])
train_exp_idxs  = np.sort(unique_exp_idx[n_eval:])

eval_mask  = np.isin(df_ann['FileIndex'].values, eval_exp_idxs)
train_mask = np.isin(df_ann['FileIndex'].values, train_exp_idxs)

print(f"Experiments : {n} total → {len(train_exp_idxs)} train / {len(eval_exp_idxs)} eval")
print(f"Particles   : {train_mask.sum():,} train / {eval_mask.sum():,} eval")

Experiments : 252 total → 215 train / 37 eval
Particles   : 33,120,888 train / 4,502,235 eval


## 3 · Build and save splits

In [5]:
OUTPUT_COLS = ['FileIndex', 'FileName', 'FileParticleIndex',
               'BinIndex', 'BinCenter', 'AnnotationValue']

train_df = df_ann.loc[train_mask].copy()
train_df['BinIndex']        = 0
train_df['BinCenter']       = 'train_particle'
train_df['AnnotationValue'] = 'train_particle'
train_df[OUTPUT_COLS].to_parquet(TRAIN_OUT, index=False)
print(f"Train split saved → {TRAIN_OUT}")

eval_df = df_ann.loc[eval_mask].copy()
eval_df['BinIndex']        = 1
eval_df['BinCenter']       = 'eval_particle'
eval_df['AnnotationValue'] = 'eval_particle'
eval_df[OUTPUT_COLS].to_parquet(EVAL_OUT, index=False)
print(f"Eval split saved  → {EVAL_OUT}")

Train split saved → ./metadata/parquet_files/train_ue.parquet
Eval split saved  → ./metadata/parquet_files/eval_ue.parquet
